Libraries

In [ ]:
import yfinance as yf
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

Initial setup

In [ ]:
tickers = pd.read_csv(
    "asx100_tickers.csv"
)["ticker"].tolist()

data = yf.download(
    tickers,
    start="2015-01-01",
    auto_adjust=True,
    group_by="ticker",
    threads=True
)

# build master data panel
dfs = []

for ticker in tickers:
    try:
        temp = data[ticker].copy()
        temp["Ticker"] = ticker
        temp = temp.reset_index()
        dfs.append(temp)

    except:
        print(f"Missing {ticker}")

panel = pd.concat(dfs)



Feature Engineering:

In [ ]:
# Momentum and reversal features
panel["mom_1m"] = panel.groupby("Ticker")["close"].pct_change(21)
panel["mom_3m"] = panel.groupby("Ticker")["close"].pct_change(63)
panel["mom_6m"] = panel.groupby("Ticker")["close"].pct_change(126)
panel["mom_12m"] = panel.groupby("Ticker")["close"].pct_change(252)
panel["reversal_1m"] = -panel["mom_1m"]

#Daily Returns
panel["daily_return"] = (
    panel.groupby("Ticker")["close"]
    .pct_change()
)

# Volatility features
panel["vol_20"] = (
    panel.groupby("Ticker")["daily_return"]
    .rolling(20)
    .std()
    .reset_index(level=0, drop=True)
)
panel["vol_20"] = (
    panel.groupby("Ticker")["daily_return"]
    .rolling(20)
    .std()
    .reset_index(level=0, drop=True)
)
panel["vol_60"] = (
    panel.groupby("Ticker")["daily_return"]
    .rolling(60)
    .std()
    .reset_index(level=0, drop=True)
)
#Ratio this is short term to long term volatility
panel["vol_ratio"] = (
    panel["vol_20"] /
    panel["vol_60"]
)

#Liquidity
panel["dollar_volume"] = (
    panel["close"] *
    panel["volume"]
)
panel["dollar_volume"] = (
    panel["close"] *
    panel["volume"]
)
avg_volume = (
    panel.groupby("Ticker")["volume"]
    .rolling(20)
    .mean()
    .reset_index(level=0, drop=True)
)

panel["volume_shock"] = (
    panel["volume"] /
    avg_volume
)
# Illiquidity
panel["amihud"] = (
    panel["daily_return"].abs() /
    panel["dollar_volume"]
)
